# SmartEdu AI - Notebook 1: Đọc và làm sạch CSV

Notebook này kiểm tra dữ liệu học tập dạng longitudinal theo từng sinh viên và học kỳ, ghi nhận lỗi dữ liệu, xử lý theo quy tắc và xuất CSV sạch cho notebook phân tích. Chạy notebook `02_analysis_A_B.ipynb` sau notebook này.

Dữ liệu đầu vào: `data/academic_survival_longitudinal.csv`. Đơn vị một dòng là một sinh viên trong một học kỳ; `Student_ID` lặp lại qua các học kỳ là bình thường, không phải dòng trùng.

## Quy tắc làm sạch và lý do

- Xóa dòng trùng lặp hoàn toàn; không xóa các học kỳ khác nhau của cùng sinh viên.
- Kiểm tra khóa `(Student_ID, Semester)` để phát hiện bản ghi sinh viên-học kỳ bị lặp.
- `Attendance` chỉ hợp lệ trong [0, 100]. Giá trị ngoài miền được chuyển thành thiếu, không tùy ý cắt về 0/100 vì không biết giá trị thật.
- `Sem_GPA` nằm trong [0, 4]; các biến đếm/chi phí/thu nhập/số người/số giờ không âm; `Work_Hours` tối đa 40 giờ/tuần. Giá trị vi phạm (nếu có) cũng được chuyển thành thiếu.
- Điền khuyết biến số bằng median của cột (ít nhạy với ngoại lệ hơn mean), biến phân loại bằng mode. Không điền khuyết nhãn; dòng thiếu nhãn bị loại bỏ vì không dùng được cho học có giám sát.
- CSV đã làm sạch giữ nguyên cột ID, nhãn và thông tin trạng thái để đối chiếu. Notebook mô hình sẽ tách ID và loại cột rò rỉ nhãn.

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SOURCE_PATH = PROJECT_ROOT / 'data' / 'academic_survival_longitudinal.csv'
CLEAN_PATH = PROJECT_ROOT / 'data' / 'academic_survival_cleaned.csv'
if not SOURCE_PATH.exists():
    raise FileNotFoundError(f'Không tìm thấy CSV đầu vào: {SOURCE_PATH}')

students_raw = pd.read_csv(SOURCE_PATH)
print(f'Đường dẫn: {SOURCE_PATH}')
print(f'Kích thước ban đầu: {students_raw.shape[0]:,} dòng x {students_raw.shape[1]} cột')
display(students_raw.head())

Đường dẫn: e:\FPT\HocKy4\MATHEMATICS_FOR_ML\ITA201_Assigment\SMARTEDU-AI\data\academic_survival_longitudinal.csv
Kích thước ban đầu: 79,239 dòng x 22 cột


,Student_ID,Age,Gender,First_Generation,Family_Income,Household_Size,Housing_Status,Scholarship,Tuition_Base,Semester,...,Emergency_Expense,Sem_GPA,Attendance,LMS_Logins,Advising_Visits,Failed_Courses,Financial_Stress,Target_Dropout_Next_Sem,End_of_Semester_Status,Censored
0,STU_00001,18,Female,0,15000.0,2,Off-Campus,0,15000,1,...,0.0,2.85,85.8,67.0,0,0,1.7,0,Enrolled,0
1,STU_00001,18,Female,0,15000.0,2,Off-Campus,0,15000,2,...,0.0,2.73,81.0,89.0,1,0,3.1,0,Enrolled,0
2,STU_00001,18,Female,0,15000.0,2,Off-Campus,0,15000,3,...,0.0,2.60,78.3,60.0,0,2,3.0,0,Enrolled,0
3,STU_00001,18,Female,0,15000.0,2,Off-Campus,0,15000,4,...,0.0,2.34,77.2,59.0,2,3,5.1,0,Enrolled,0
4,STU_00001,18,Female,0,15000.0,2,Off-Campus,0,15000,5,...,0.0,1.90,79.3,53.0,1,2,3.6,0,Enrolled,1


In [6]:
print('Kiểu dữ liệu:')
display(students_raw.dtypes.rename('Kiểu dữ liệu').to_frame())

missing_report = students_raw.isna().sum().rename('Số giá trị thiếu').to_frame()
missing_report['Tỷ lệ thiếu (%)'] = (missing_report['Số giá trị thiếu'] / len(students_raw) * 100).round(2)
print('Các cột có giá trị thiếu:')
display(missing_report.loc[missing_report['Số giá trị thiếu'].gt(0)])

exact_duplicates = int(students_raw.duplicated().sum())
key_duplicates = int(students_raw.duplicated(['Student_ID', 'Semester']).sum())
print(f'Dòng trùng lặp hoàn toàn: {exact_duplicates:,}')
print(f'Khóa Student_ID-Semester bị lặp: {key_duplicates:,}')
print(f'Số Student_ID duy nhất: {students_raw.Student_ID.nunique():,}')
print('Số dòng theo học kỳ:')
display(students_raw['Semester'].value_counts().sort_index().rename('Số bản ghi').to_frame())

Kiểu dữ liệu:


,Kiểu dữ liệu
Student_ID,str
Age,int64
Gender,str
First_Generation,int64
Family_Income,float64
Household_Size,int64
Housing_Status,str
Scholarship,int64
Tuition_Base,int64
Semester,int64


Các cột có giá trị thiếu:


,Số giá trị thiếu,Tỷ lệ thiếu (%)
Family_Income,3604,4.55
LMS_Logins,867,1.09


Dòng trùng lặp hoàn toàn: 0
Khóa Student_ID-Semester bị lặp: 0
Số Student_ID duy nhất: 20,000
Số dòng theo học kỳ:


,Số bản ghi
Semester,
1,20000
2,16596
3,13231
4,10239
5,7705
6,5554
7,3742
8,2172


In [7]:
# Quy tắc miền dữ liệu dựa trên ý nghĩa cột và đơn vị ghi trong dataset.
domain_rules = {
    'Attendance': (0, 100),
    'Sem_GPA': (0, 4),
    'Work_Hours': (0, 40),
    'Family_Income': (0, None),
    'Emergency_Expense': (0, None),
    'Household_Size': (1, None),
    'Course_Load': (0, None),
    'Tuition_Base': (0, None),
    'LMS_Logins': (0, None),
    'Advising_Visits': (0, None),
    'Failed_Courses': (0, None),
    'Financial_Stress': (0, None),
}
range_rows = []
for column, (lower, upper) in domain_rules.items():
    values = students_raw[column]
    invalid = pd.Series(False, index=values.index)
    if lower is not None:
        invalid |= values.lt(lower)
    if upper is not None:
        invalid |= values.gt(upper)
    range_rows.append({'Đặc trưng': column, 'Giá trị nhỏ nhất': values.min(), 'Giá trị lớn nhất': values.max(), 'Số giá trị ngoài miền': int(invalid.sum())})
range_report = pd.DataFrame(range_rows).set_index('Đặc trưng')
display(range_report)

binary_columns = ['First_Generation', 'Scholarship', 'Target_Dropout_Next_Sem', 'Censored']
binary_errors = {column: int((~students_raw[column].isin([0, 1])).sum()) for column in binary_columns}
print('Số giá trị ngoài {0,1} ở các cột nhị phân:', binary_errors)

,Giá trị nhỏ nhất,Giá trị lớn nhất,Số giá trị ngoài miền
Đặc trưng,,,
Attendance,-2.00,105.00,249
Sem_GPA,0.44,3.96,0
Work_Hours,0.00,40.00,0
Family_Income,3000.00,488000.00,0
Emergency_Expense,0.00,3998.00,0
Household_Size,1.00,6.00,0
Course_Load,9.00,21.00,0
Tuition_Base,15000.00,45000.00,0
LMS_Logins,12.00,116.00,0


Số giá trị ngoài {0,1} ở các cột nhị phân: {'First_Generation': 0, 'Scholarship': 0, 'Target_Dropout_Next_Sem': 0, 'Censored': 0}


## Áp dụng làm sạch

Các con số ở bảng audit là số liệu thực tế từ CSV. Sau bước này, notebook hiển thị số dòng trước/sau, số giá trị ngoài miền và số ô khuyết đã xử lý. Nếu một khóa sinh viên-học kỳ bị lặp, notebook báo lỗi thay vì tự chọn bản ghi, vì việc chọn có thể làm mất thông tin.

In [8]:
students = students_raw.drop_duplicates().copy()
if students.duplicated(['Student_ID', 'Semester']).any():
    raise ValueError('Khóa Student_ID-Semester bị lặp; cần kiểm tra thủ công trước khi xuất CSV.')

rows_before_target_check = len(students)
missing_target = int(students['Target_Dropout_Next_Sem'].isna().sum())
students = students.dropna(subset=['Target_Dropout_Next_Sem']).copy()

invalid_by_column = {}
for column, (lower, upper) in domain_rules.items():
    invalid = pd.Series(False, index=students.index)
    if lower is not None:
        invalid |= students[column].lt(lower)
    if upper is not None:
        invalid |= students[column].gt(upper)
    invalid_by_column[column] = int(invalid.sum())
    students.loc[invalid, column] = np.nan

missing_before_imputation = students.isna().sum()
numeric_columns = students.select_dtypes(include='number').columns.tolist()
categorical_columns = students.select_dtypes(exclude='number').columns.tolist()
for column in numeric_columns:
    if column != 'Target_Dropout_Next_Sem' and students[column].isna().any():
        students[column] = students[column].fillna(students[column].median())
for column in categorical_columns:
    if column != 'Student_ID' and students[column].isna().any():
        students[column] = students[column].fillna(students[column].mode().iloc[0])

students.to_csv(CLEAN_PATH, index=False)
cleaning_summary = pd.Series({
    'Số dòng ban đầu': len(students_raw),
    'Số dòng trùng hoàn toàn đã xóa': exact_duplicates,
    'Số dòng thiếu nhãn đã xóa': missing_target,
    'Số giá trị Attendance ngoài miền chuyển thành thiếu': invalid_by_column['Attendance'],
    'Tổng giá trị ngoài miền chuyển thành thiếu': sum(invalid_by_column.values()),
    'Số ô thiếu đã điền': int(missing_before_imputation.sum()),
    'Số dòng sau làm sạch': len(students),
    'Số cột': students.shape[1],
})
display(cleaning_summary.rename('Giá trị').to_frame())
print(f'CSV sạch đã được lưu tại: {CLEAN_PATH}')
print('Số giá trị thiếu còn lại:', int(students.isna().sum().sum()))
print('Kiểm tra Attendance sau làm sạch:', students['Attendance'].min(), 'đến', students['Attendance'].max())

,Giá trị
Số dòng ban đầu,79239
Số dòng trùng hoàn toàn đã xóa,0
Số dòng thiếu nhãn đã xóa,0
Số giá trị Attendance ngoài miền chuyển thành thiếu,249
Tổng giá trị ngoài miền chuyển thành thiếu,249
Số ô thiếu đã điền,4720
Số dòng sau làm sạch,79239
Số cột,22


CSV sạch đã được lưu tại: e:\FPT\HocKy4\MATHEMATICS_FOR_ML\ITA201_Assigment\SMARTEDU-AI\data\academic_survival_cleaned.csv
Số giá trị thiếu còn lại: 0
Kiểm tra Attendance sau làm sạch: 58.7 đến 100.0


## Kết luận và cách đọc kết quả

CSV là dữ liệu longitudinal: cùng một `Student_ID` có thể xuất hiện ở nhiều `Semester`; không được xóa các dòng này như bản ghi trùng. Dùng khóa ghép `Student_ID` + `Semester` để kiểm tra uniqueness. Các giá trị khuyết được điền median/mode để giữ lại quan sát, nhưng đây là quyết định phục vụ bài tập, không khẳng định giá trị điền là giá trị thật.

`Target_Dropout_Next_Sem` là nhãn nhị phân 0/1. Không dùng `Student_ID` làm feature. `End_of_Semester_Status` và `Censored` được giữ trong CSV để đối chiếu nghiệp vụ; notebook mô hình sẽ loại chúng khỏi feature vì thông tin kết quả/trạng thái có thể tiết lộ nhãn.

Sau khi chạy xong, mở `02_analysis_A_B.ipynb`. Nếu cần lặp lại kết quả, chạy notebook này từ đầu để tạo lại `data/academic_survival_cleaned.csv`.